In [1]:
! wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/N0CceRlquaf9q85PK759WQ/regression-dataset.csv
! wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7J73m6Nsz-vmojwab91gMA/classification-dataset.csv

--2026-02-04 19:19:34--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/N0CceRlquaf9q85PK759WQ/regression-dataset.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 198.23.119.245
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|198.23.119.245|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2002174 (1.9M) [text/csv]
Saving to: ‘regression-dataset.csv’

regression-dataset. 100%[===================>]   1.91M  4.70MB/s    in 0.4s    

2026-02-04 19:19:35 (4.70 MB/s) - ‘regression-dataset.csv’ saved [2002174/2002174]

--2026-02-04 19:19:35--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7J73m6Nsz-vmojwab91gMA/classification-dataset.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomai

In [11]:
!pip install scikit-learn

In [1]:
import numpy as np
import pandas as pd
import matplotlib
import seaborn
import sklearn
import langchain
import groq
import langchain_groq
import glob
import os
from typing import List , Optional


/Users/anirudh/.local/share/virtualenvs/Desktop-sFnGVMJ4/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_core.tools import tool
@tool
def list_csv_files()->Optional[List[str]]:
    """
    Lists all csv files in the directory
    Parameters:
    No input parameters
    Returns:
    It returns the list of all csv files available in the directory.
    If,there are no csv files it returns None
    """
    csv_files = glob.glob(os.path.join(os.getcwd(),"*.csv"))
    if not csv_files:
        return None
    return [os.path.basename(file) for file in csv_files]    

In [3]:
print("Tool Name:", list_csv_files.name)
print("Tool Description:", list_csv_files.description)
print("Tool Arguments:", list_csv_files.args)

Tool Name: list_csv_files
Tool Description: Lists all csv files in the directory
Parameters:
No input parameters
Returns:
It returns the list of all csv files available in the directory.
If,there are no csv files it returns None
Tool Arguments: {}


In [4]:
# Creating a data cache for tools to refer and to minimize context
DATAFRAME_CACHE = {}
@tool
def preload_datasets(paths:List[str])->str:
    """
    Loads CSV files into a global cache,if not done already.
    This function helps to efficiently manage datasets by loading them once 
    and storing in memory for future use. Without caching,you would waste tokens
    describing dataset contents repeatedly in agent responses
    Args:
        paths: A list of file paths to csv files
    Returns :
            A message summarizing which datasets were loaded or cached
    """
    loaded = []
    cached = []
    for path in paths:
        if path not in DATAFRAME_CACHE:
            DATAFRAME_CACHE[path] = pd.read_csv(path)
            loaded.append(path)
        else:
            cached.append(path)
    return (
        f"Loaded datasets: {loaded}\n"
        f"Already cached: {cached}"
    )        
        
     

In [5]:
#Summarization tool
from typing import List,Dict,Optional,Any
@tool
def get_dataset_summaries(dataset_paths:List[str])->List[Dict[str,Any]] :
    """
    Analyze multiple csv files and return metadata summaries for each.
    Args:
    -dataset_paths (List[str]):
            A list of file paths to csv files
   Returns:
   -List[Dict[str,Any]]:
       A list of summaries, one per dataset, each containing:
            - "file_name": The path of the dataset file.
            - "column_names": A list of column names in the dataset.
            - "data_types": A dictionary mapping column names to their data types (as strings).
    """
    summaries = []
    for path in dataset_paths:
        #Load and cache the data set if not cached
        if path not in DATAFRAME_CACHE:
            DATAFRAME_CACHE[path] = pd.read_csv(path)
        df = DATAFRAME_CACHE[path]
        #Build Summary
        summary ={
            "file_name":path,
            "column_names": df.columns.to_list(),
            "data_types": df.dtypes.astype(str).to_dict()
        }
        summaries.append(summary)
    return summaries    
            

In [6]:
@tool
def call_dataframe_method(file_name:str,method:str)->str:
    """
    This function executes a method on a dataframe and returns the result.
    This tool lets you run simple DataFrame methods like 'head', 'tail', or 'describe' 
   on a dataset that has already been loaded and cached using 'preload_datasets'.
    Args:
       file_name (str): The path or name of the dataset in the global cache.
       method (str): The name of the method to call on the DataFrame. Only no-argument 
                     methods are supported (e.g., 'head', 'describe', 'info').
    Returns:
       str: The output of the method as a formatted string, or an error message if 
            the dataset is not found or the method is invalid.
    Example:
       call_dataframe_method(file_name="data.csv", method="head")
    """
    if file_name not in DATAFRAME_CACHE:
       try:
           DATAFRAME_CACHE[file_name] = pd.read_csv(file_name)
       except FileNotFoundError:
           return f"Data frame {file_name} not found in cache or disk "
       except Exception as e:
           return f"Error loading {file_name} :{str(e)}"
    df = DATAFRAME_CACHE[file_name]
    func = getattr(df, method, None)
    if not callable(func):
       return f"{method} isn't a valid method on dataframe"
    try:
       result = func()
       return str(result)
    except Exception as e:
       return f"Error calling '{method}' on '{file_name}' : {str(e)}"
           

In [26]:
#Model evaluation tools
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier ,RandomForestRegressor
from sklearn.metrics import accuracy_score, r2_score , mean_squared_error

DATAFRAME_CACHE ={}
@tool
def evaluate_classification_dataset(file_name:str , target_column:str)->Dict[str, float] :
    """
    Trains and evaluates a classifier on a dataset using specified target column
    Args:
        file_name(str):The name of the path stored in DATAFRAME_CACHE
        target_column(str):The name of the column to use as classification target
    Returns:
    Dict[str,float]: A dictionary with the model's accuracy score
    """
    if file_name not in DATAFRAME_CACHE:
        try:
            DATAFRAME_CACHE[file_name] = pd.read_csv(file_name)
        except FileNotFoundError:
            return {"error":f"Dataframe '{file_name}' not found in cache or disk"}
        except Exception as e:
            return {"error": f"Error loading '{file_name}':{str(e)}"}
    df = DATAFRAME_CACHE[file_name]
    if target_column not in df.columns:
        return {"error": f"Target column '{target_column}' not found in '{file_name}'"}
    X = df.drop(columns=[target_column])
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42) 
    model = RandomForestClassifier()
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return {"result":acc}
@tool
def evaluate_regression_dataset(file_name:str, target_column:str)-> Dict[str,float]:
    """
     Train and evaluate a regression model on a dataset using the specified target column.
    Args:
        file_name (str): The name or path of the dataset stored in DATAFRAME_CACHE.
        target_column (str): The name of the column to use as the regression target.
    Returns:
        Dict[str, float]: A dictionary with R² score and Mean Squared Error.
    """
    if file_name not in DATAFRAME_CACHE:
        try:
           DATAFRAME_CACHE[file_name] = pd.read_csv(file_name)
        except FileNotFoundError:
            return {"error":f"Dataframe '{file_name}' not found in cache or disk"}
        except Exception as e:
            return {"error": f"Error loading '{file_name}':{str(e)}"}
    df = DATAFRAME_CACHE[file_name]
    if target_column not in df.columns:
        return {"error": f"Target column '{target_column_}' not found in '{file_name}'"}
    X = df.drop(columns = [target_column])
    y = df[target_column]
    X_train,X_test ,y_train, y_test = train_test_split(X, y , test_size = 0.2, random_state = 42)
    model = RandomForestRegressor()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test , y_pred)
    return {"r2_score":r2,"mean_square_error":mse}
           
        

In [27]:
from langchain_groq import ChatGroq
from langchain_classic.agents import create_tool_calling_agent,AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
     ("system", 
     "You are a data science assistant. Use the available tools to analyze CSV files. "
     "Your job is to determine whether each dataset is for classification or regression, based on its structure."),
    
    ("user", "{input}"),
    ("placeholder", "{agent_scratchpad}")  # Required for tool-calling agents
    ]
)

In [28]:
from langchain.chat_models import init_chat_model
llm = init_chat_model("llama-3.3-70B-versatile", model_provider="groq", api_key = "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", streaming=False )

In [29]:
tools=[list_csv_files, preload_datasets, get_dataset_summaries, call_dataframe_method, evaluate_classification_dataset, evaluate_regression_dataset]

In [30]:
agent = create_tool_calling_agent(llm, tools, prompt)

In [32]:
agent_executor = AgentExecutor(agent=agent , tools= tools ,verbose=True , handle_parsing_errors= True)

In [33]:
agent_executor.agent.stream_runnable = False

In [ ]:
print("📊 Ask questions about your dataset (type 'exit' to quit):")

while True:
    user_input=input(" You:")
    if user_input.strip().lower() in ['exit','quit']:
        print("see ya later")
        break
        
    result=agent_executor.invoke({"input":user_input})
    print(f"my Agent: {result['output']}")

📊 Ask questions about your dataset (type 'exit' to quit):


 You: What are the datasets?.




> Entering new AgentExecutor chain...

Invoking: `list_csv_files` with `{}`


['classification-dataset.csv', 'regression-dataset.csv']
Invoking: `get_dataset_summaries` with `{'dataset_paths': ['classification-dataset.csv', 'regression-dataset.csv']}`


[{'file_name': 'classification-dataset.csv', 'column_names': ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension', 'target'], 'data_types': {'mean radius': 'float64', 'mean texture': 'float64', 'mean perimeter': 'float64',